# 🎬 Supernan Video Dubbing Pipeline
### Kannada → Hindi | Sarvam AI + Open Source | Google Colab T4

This notebook runs the complete dubbing pipeline:
1. **Extract** a 15s clip from the source video
2. **Transcribe** Kannada audio (Sarvam AI Saaras V3)
3. **Translate** Kannada → Hindi (Sarvam AI Translate)
4. **Voice Clone** Hindi speech (Coqui XTTSv2)
5. **Lip Sync** video (Wav2Lip)
6. **Face Restore** (CodeFormer)
7. **Final Output** dubbed video

---
**⚠️ Requirements:**
- Colab runtime with **GPU (T4)** — Go to `Runtime > Change runtime type > T4 GPU`
- A **Sarvam AI API key** — [Get one free](https://dashboard.sarvam.ai/)

## 🔧 Cell 1: GPU Check & System Setup

In [ ]:
# ── Verify GPU availability ──────────────────────────────────────────
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ No GPU detected! Go to Runtime > Change runtime type > T4 GPU"
    )
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB VRAM)")

# ── Install system dependencies ──────────────────────────────────────
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1
!ffmpeg -version | head -1
print("✅ ffmpeg installed")

## 📦 Cell 2: Clone Repos & Install Dependencies

In [ ]:
import os

# ── Clone the project repo ────────────────────────────────────────────
PROJECT_DIR = "/content/Supernan-Video-Dubbing-System"

if not os.path.exists(PROJECT_DIR):
    # Option A: Clone from GitHub (if you've pushed the code)
    # !git clone https://github.com/YOUR_USERNAME/Supernan-Video-Dubbing-System.git {PROJECT_DIR}

    # Option B: Upload from local (if running from Drive)
    # Mount Google Drive and copy:
    from google.colab import drive
    drive.mount('/content/drive')
    !cp -r "/content/drive/MyDrive/Supernan-Video-Dubbing-System" {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print(f"✅ Working directory: {os.getcwd()}")
!ls -la

In [ ]:
# ── Install Python dependencies ──────────────────────────────────────
!pip install -q -r requirements.txt
print("✅ pip dependencies installed")

In [ ]:
# ── Clone Wav2Lip ────────────────────────────────────────────────────
WAV2LIP_DIR = os.path.join(PROJECT_DIR, "Wav2Lip")
if not os.path.exists(WAV2LIP_DIR):
    !git clone https://github.com/Rudrabha/Wav2Lip.git {WAV2LIP_DIR}
    # Install Wav2Lip dependencies
    !pip install -q -r {WAV2LIP_DIR}/requirements.txt
print(f"✅ Wav2Lip cloned: {WAV2LIP_DIR}")

# ── Download Wav2Lip checkpoints ─────────────────────────────────────
CHECKPOINTS_DIR = os.path.join(WAV2LIP_DIR, "checkpoints")
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

WAV2LIP_GAN_URL = "https://iiitaphyd-my.sharepoint.com/:u:/g/personal/radrabha_m_research_iiit_ac_in/Eb3LEzbfuKlJiR600x9zSOsBIrHq8JXYEYH1M7sPcqTKwA?download=1"
WAV2LIP_GAN_PATH = os.path.join(CHECKPOINTS_DIR, "wav2lip_gan.pth")

if not os.path.exists(WAV2LIP_GAN_PATH):
    print("Downloading wav2lip_gan.pth (~150 MB)...")
    !wget -q -O {WAV2LIP_GAN_PATH} "{WAV2LIP_GAN_URL}"
print(f"✅ Wav2Lip GAN checkpoint: {WAV2LIP_GAN_PATH}")

# Download face detection model for Wav2Lip
FACE_DET_URL = "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth"
FACE_DET_DIR = os.path.join(WAV2LIP_DIR, "face_detection", "detection", "sfd")
FACE_DET_PATH = os.path.join(FACE_DET_DIR, "s3fd.pth")
os.makedirs(FACE_DET_DIR, exist_ok=True)

if not os.path.exists(FACE_DET_PATH):
    print("Downloading face detection model...")
    !wget -q -O {FACE_DET_PATH} "{FACE_DET_URL}"
print(f"✅ Face detection model: {FACE_DET_PATH}")

In [ ]:
# ── Clone CodeFormer ─────────────────────────────────────────────────
CODEFORMER_DIR = os.path.join(PROJECT_DIR, "CodeFormer")
if not os.path.exists(CODEFORMER_DIR):
    !git clone https://github.com/sczhou/CodeFormer.git {CODEFORMER_DIR}
    # Install CodeFormer dependencies
    !pip install -q -r {CODEFORMER_DIR}/requirements.txt
    %cd {CODEFORMER_DIR}
    !python basicsr/setup.py develop > /dev/null 2>&1
    %cd {PROJECT_DIR}
print(f"✅ CodeFormer cloned: {CODEFORMER_DIR}")

# Download CodeFormer pre-trained weights
CF_WEIGHTS_DIR = os.path.join(CODEFORMER_DIR, "weights", "CodeFormer")
os.makedirs(CF_WEIGHTS_DIR, exist_ok=True)
CF_WEIGHT_PATH = os.path.join(CF_WEIGHTS_DIR, "codeformer.pth")

if not os.path.exists(CF_WEIGHT_PATH):
    CF_WEIGHT_URL = "https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/codeformer.pth"
    print("Downloading CodeFormer weights...")
    !wget -q -O {CF_WEIGHT_PATH} "{CF_WEIGHT_URL}"

# Download face parsing model for CodeFormer
FACELIB_DIR = os.path.join(CODEFORMER_DIR, "weights", "facelib")
os.makedirs(FACELIB_DIR, exist_ok=True)

FACELIB_MODELS = {
    "detection_Resnet50_Final.pth": "https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/detection_Resnet50_Final.pth",
    "parsing_parsenet.pth": "https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/parsing_parsenet.pth",
}

for fname, url in FACELIB_MODELS.items():
    fpath = os.path.join(FACELIB_DIR, fname)
    if not os.path.exists(fpath):
        print(f"Downloading {fname}...")
        !wget -q -O {fpath} "{url}"

print("✅ CodeFormer weights downloaded")

## 🔑 Cell 3: Set Sarvam AI API Key

In [ ]:
# ── Set your Sarvam AI API key ───────────────────────────────────────
# Get your free key at: https://dashboard.sarvam.ai/

import os
from google.colab import userdata

# Option 1: Set via Colab Secrets (recommended)
# Go to the 🔑 icon in the left sidebar > Add secret: SARVAM_API_KEY
try:
    SARVAM_API_KEY = userdata.get('SARVAM_API_KEY')
    print("✅ API key loaded from Colab Secrets")
except Exception:
    # Option 2: Paste it here directly (less secure)
    SARVAM_API_KEY = "YOUR_API_KEY_HERE"  # ← Replace this
    print("⚠️  Using hardcoded API key (use Colab Secrets instead)")

os.environ["SARVAM_API_KEY"] = SARVAM_API_KEY

if SARVAM_API_KEY == "YOUR_API_KEY_HERE":
    raise ValueError("❌ Please set your Sarvam AI API key! See instructions above.")

print(f"✅ API key set (length: {len(SARVAM_API_KEY)})")

## 📥 Cell 4: Download Source Video

In [ ]:
# ── Download the source Kannada video from Google Drive ──────────────
import os
os.chdir(PROJECT_DIR)

INPUT_DIR = os.path.join(PROJECT_DIR, "input")
os.makedirs(INPUT_DIR, exist_ok=True)

SOURCE_VIDEO = os.path.join(INPUT_DIR, "source.mp4")

if not os.path.exists(SOURCE_VIDEO):
    # Google Drive file ID from the shared link
    GDRIVE_FILE_ID = "1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW"
    print("Downloading source video from Google Drive...")
    !pip install -q gdown
    !gdown --id {GDRIVE_FILE_ID} -O {SOURCE_VIDEO}

# Verify download
file_size = os.path.getsize(SOURCE_VIDEO) / 1024 / 1024
print(f"✅ Source video: {SOURCE_VIDEO} ({file_size:.1f} MB)")

# Quick preview info
!ffprobe -v error -show_entries format=duration -of default=noprint_wrappers=1:nokey=1 {SOURCE_VIDEO}

## ▶️ Cell 5: Run Tests (Sanity Check)

In [ ]:
# ── Run the test suite to verify everything is set up ────────────────
os.chdir(PROJECT_DIR)
!python -m pytest tests/ -v --tb=short

## 🚀 Cell 6: Run the Full Dubbing Pipeline

This cell runs all 7 steps sequentially.

**Adjust the timestamp range** below if you want a different 15-second segment.

In [ ]:
# ── Configure the pipeline ───────────────────────────────────────────
START_TIME = 15   # Start of clip in seconds
END_TIME = 30     # End of clip in seconds
FIDELITY_WEIGHT = 0.7  # CodeFormer: 0.0=quality, 1.0=faithful

# ── Run the pipeline ─────────────────────────────────────────────────
os.chdir(PROJECT_DIR)
!python dub_video.py \
    --input input/source.mp4 \
    --start {START_TIME} \
    --end {END_TIME} \
    --output-dir output \
    --fidelity-weight {FIDELITY_WEIGHT} \
    --sarvam-api-key {SARVAM_API_KEY}

## 📝 Cell 7 (Optional): Review & Correct Transcript

If the Kannada transcription needs correction:
1. Edit `output/temp/transcript.txt`
2. Re-run with `--skip-transcribe` flag

In [ ]:
# ── View the transcript ──────────────────────────────────────────────
print("=" * 50)
print("Kannada Transcript:")
print("=" * 50)
!cat output/temp/transcript.txt

print("\n" + "=" * 50)
print("Hindi Translation:")
print("=" * 50)
!cat output/temp/hindi_text.txt

In [ ]:
# ── (Optional) Edit transcript and re-run ────────────────────────────
# Uncomment and modify the text below, then run this cell

# CORRECTED_TRANSCRIPT = "your corrected Kannada text here"
# with open("output/temp/transcript.txt", "w") as f:
#     f.write(CORRECTED_TRANSCRIPT)

# # Re-run pipeline skipping transcription (uses your corrected text)
# !python dub_video.py \
#     --input input/source.mp4 \
#     --start {START_TIME} \
#     --end {END_TIME} \
#     --output-dir output \
#     --skip-transcribe \
#     --sarvam-api-key {SARVAM_API_KEY}

## 🎥 Cell 8: Preview & Download Output

In [ ]:
# ── Preview the final dubbed video ───────────────────────────────────
from IPython.display import HTML, Video
import os

final_video = os.path.join(PROJECT_DIR, "output", "final_output.mp4")

if os.path.exists(final_video):
    file_size = os.path.getsize(final_video) / 1024 / 1024
    print(f"✅ Final video: {final_video} ({file_size:.1f} MB)")
    !ffprobe -v error -show_entries format=duration -of default=noprint_wrappers=1:nokey=1 {final_video}
    
    # Display video inline
    display(Video(final_video, embed=True, width=640))
else:
    print("❌ Final video not found. Check pipeline logs above for errors.")

In [ ]:
# ── Download the final video ─────────────────────────────────────────
from google.colab import files

if os.path.exists(final_video):
    files.download(final_video)
    print("✅ Download started!")
else:
    print("❌ No video to download.")

## 🔍 Cell 9: Side-by-Side Comparison

In [ ]:
# ── Compare original clip vs dubbed clip ─────────────────────────────
from IPython.display import HTML

original_clip = os.path.join(PROJECT_DIR, "output", "temp", "clip.mp4")
dubbed_clip = os.path.join(PROJECT_DIR, "output", "final_output.mp4")

if os.path.exists(original_clip) and os.path.exists(dubbed_clip):
    print("Original (Kannada):")
    display(Video(original_clip, embed=True, width=480))
    print("\nDubbed (Hindi):")
    display(Video(dubbed_clip, embed=True, width=480))
else:
    print("Run the pipeline first (Cell 6).")